<a href="https://colab.research.google.com/github/apsarak26/capstone_project_CS5998/blob/main/Outcome_labeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import pandas as pd

input_path = "/content/drive/MyDrive/allergy_candidate_terms.csv"

candidate_df = pd.read_csv(input_path)

print("Original candidate terms:", len(candidate_df))
print("Columns:", candidate_df.columns.tolist())

display(candidate_df.head())


Original candidate terms: 197
Columns: ['pt']


,pt
0,ALLERGENIC DESENSITISATION PROCEDURE
1,ALLERGIC BRONCHITIS
2,ALLERGIC COLITIS
3,ALLERGIC COUGH
4,ALLERGIC CYSTITIS


In [6]:
review_df = candidate_df[["pt"]].copy()

# Remove missing or empty terms
review_df = review_df.dropna(subset=["pt"])

# Standardise terms for matching
review_df["pt_normalized"] = (
    review_df["pt"]
    .astype(str)
    .str.strip()
    .str.upper()
)

review_df = review_df[
    review_df["pt_normalized"] != ""
]

# Remove duplicates after normalisation
review_df = review_df.drop_duplicates(
    subset=["pt_normalized"]
).reset_index(drop=True)

print("Original candidates:", len(candidate_df))
print("Unique normalised terms:", len(review_df))


Original candidates: 197
Unique normalised terms: 111


In [7]:
core_terms = {
    "ANAPHYLACTIC REACTION",
    "ANAPHYLACTIC SHOCK",
    "ANAPHYLACTOID REACTION",
    "ANAPHYLACTOID SHOCK",
    "DOCUMENTED HYPERSENSITIVITY TO ADMINISTERED DRUG",
    "DRUG HYPERSENSITIVITY",
    "HYPERSENSITIVITY",
    "TYPE I HYPERSENSITIVITY",
    "TYPE II HYPERSENSITIVITY",
    "TYPE IV HYPERSENSITIVITY REACTION"
}

def assign_screening_group(term):
    if term in core_terms:
        return "CORE_CANDIDATE"

    if any(keyword in term for keyword in [
        "TEST", "HISTORY", "FAMILY HISTORY",
        "SEASONAL ALLERGY", "FOOD ALLERGY",
        "LATEX ALLERGY", "TREATMENT"
    ]):
        return "CONTEXTUAL_OR_OTHER"

    return "EXPANDED_OR_AMBIGUOUS"

review_df["screening_group"] = (
    review_df["pt_normalized"]
    .apply(assign_screening_group)
)

print(review_df["screening_group"].value_counts())


screening_group
EXPANDED_OR_AMBIGUOUS    93
CORE_CANDIDATE           10
CONTEXTUAL_OR_OTHER       8
Name: count, dtype: int64


In [8]:
review_df["final_decision"] = "PENDING"
review_df["reference"] = ""
review_df["review_notes"] = ""

review_df = review_df[
    [
        "pt",
        "pt_normalized",
        "screening_group",
        "final_decision",
        "reference",
        "review_notes"
    ]
]

display(review_df.head(20))


,pt,pt_normalized,screening_group,final_decision,reference,review_notes
0,ALLERGENIC DESENSITISATION PROCEDURE,ALLERGENIC DESENSITISATION PROCEDURE,EXPANDED_OR_AMBIGUOUS,PENDING,,
1,ALLERGIC BRONCHITIS,ALLERGIC BRONCHITIS,EXPANDED_OR_AMBIGUOUS,PENDING,,
2,ALLERGIC COLITIS,ALLERGIC COLITIS,EXPANDED_OR_AMBIGUOUS,PENDING,,
3,ALLERGIC COUGH,ALLERGIC COUGH,EXPANDED_OR_AMBIGUOUS,PENDING,,
4,ALLERGIC CYSTITIS,ALLERGIC CYSTITIS,EXPANDED_OR_AMBIGUOUS,PENDING,,
5,ALLERGIC GRANULOMATOUS ANGIITIS,ALLERGIC GRANULOMATOUS ANGIITIS,EXPANDED_OR_AMBIGUOUS,PENDING,,
6,ALLERGIC HEPATITIS,ALLERGIC HEPATITIS,EXPANDED_OR_AMBIGUOUS,PENDING,,
7,ALLERGIC KERATITIS,ALLERGIC KERATITIS,EXPANDED_OR_AMBIGUOUS,PENDING,,
8,ALLERGIC MYOCARDITIS,ALLERGIC MYOCARDITIS,EXPANDED_OR_AMBIGUOUS,PENDING,,
9,ALLERGIC OEDEMA,ALLERGIC OEDEMA,EXPANDED_OR_AMBIGUOUS,PENDING,,


In [9]:
output_path = (
    "/content/drive/MyDrive/"
    "aeolus_hypersensitivity_review_template.csv"
)

review_df.to_csv(output_path, index=False)

print("Review template saved successfully!")
print("Location:", output_path)
print("Total terms:", len(review_df))

print("\nScreening group distribution:")
print(review_df["screening_group"].value_counts())


Review template saved successfully!
Location: /content/drive/MyDrive/aeolus_hypersensitivity_review_template.csv
Total terms: 111

Screening group distribution:
screening_group
EXPANDED_OR_AMBIGUOUS    93
CORE_CANDIDATE           10
CONTEXTUAL_OR_OTHER       8
Name: count, dtype: int64


In [10]:
core_df = review_df[
    review_df["screening_group"] == "CORE_CANDIDATE"
]

print("Core candidates:", len(core_df))

display(
    core_df[
        [
            "pt_normalized",
            "final_decision",
            "reference"
        ]
    ]
)


Core candidates: 10


,pt_normalized,final_decision,reference
28,ANAPHYLACTIC REACTION,PENDING,
29,ANAPHYLACTIC SHOCK,PENDING,
31,ANAPHYLACTOID REACTION,PENDING,
32,ANAPHYLACTOID SHOCK,PENDING,
54,DOCUMENTED HYPERSENSITIVITY TO ADMINISTERED DRUG,PENDING,
55,DRUG HYPERSENSITIVITY,PENDING,
67,HYPERSENSITIVITY,PENDING,
95,TYPE I HYPERSENSITIVITY,PENDING,
96,TYPE II HYPERSENSITIVITY,PENDING,
97,TYPE IV HYPERSENSITIVITY REACTION,PENDING,


In [11]:
import zipfile
from pathlib import Path

review_path = "/content/drive/MyDrive/aeolus_hypersensitivity_review_template.csv"
zip_path = "/content/drive/MyDrive/aeolus_v1.zip"

review_df = pd.read_csv(review_path, keep_default_na=False)

print("Total terms:", len(review_df))
print("\nScreening groups:")
print(review_df["screening_group"].value_counts())

display(review_df.head(10))

Total terms: 111

Screening groups:
screening_group
EXPANDED_OR_AMBIGUOUS    93
CORE_CANDIDATE           10
CONTEXTUAL_OR_OTHER       8
Name: count, dtype: int64


,pt,pt_normalized,screening_group,final_decision,reference,review_notes
0,ALLERGENIC DESENSITISATION PROCEDURE,ALLERGENIC DESENSITISATION PROCEDURE,EXPANDED_OR_AMBIGUOUS,PENDING,,
1,ALLERGIC BRONCHITIS,ALLERGIC BRONCHITIS,EXPANDED_OR_AMBIGUOUS,PENDING,,
2,ALLERGIC COLITIS,ALLERGIC COLITIS,EXPANDED_OR_AMBIGUOUS,PENDING,,
3,ALLERGIC COUGH,ALLERGIC COUGH,EXPANDED_OR_AMBIGUOUS,PENDING,,
4,ALLERGIC CYSTITIS,ALLERGIC CYSTITIS,EXPANDED_OR_AMBIGUOUS,PENDING,,
5,ALLERGIC GRANULOMATOUS ANGIITIS,ALLERGIC GRANULOMATOUS ANGIITIS,EXPANDED_OR_AMBIGUOUS,PENDING,,
6,ALLERGIC HEPATITIS,ALLERGIC HEPATITIS,EXPANDED_OR_AMBIGUOUS,PENDING,,
7,ALLERGIC KERATITIS,ALLERGIC KERATITIS,EXPANDED_OR_AMBIGUOUS,PENDING,,
8,ALLERGIC MYOCARDITIS,ALLERGIC MYOCARDITIS,EXPANDED_OR_AMBIGUOUS,PENDING,,
9,ALLERGIC OEDEMA,ALLERGIC OEDEMA,EXPANDED_OR_AMBIGUOUS,PENDING,,


In [12]:
core_df = review_df[
    review_df["screening_group"] == "CORE_CANDIDATE"
].copy()

display(core_df[
    ["pt_normalized", "final_decision", "reference", "review_notes"]
])

,pt_normalized,final_decision,reference,review_notes
28,ANAPHYLACTIC REACTION,PENDING,,
29,ANAPHYLACTIC SHOCK,PENDING,,
31,ANAPHYLACTOID REACTION,PENDING,,
32,ANAPHYLACTOID SHOCK,PENDING,,
54,DOCUMENTED HYPERSENSITIVITY TO ADMINISTERED DRUG,PENDING,,
55,DRUG HYPERSENSITIVITY,PENDING,,
67,HYPERSENSITIVITY,PENDING,,
95,TYPE I HYPERSENSITIVITY,PENDING,,
96,TYPE II HYPERSENSITIVITY,PENDING,,
97,TYPE IV HYPERSENSITIVITY REACTION,PENDING,,


In [13]:
provisional_terms = set(
    core_df["pt_normalized"]
    .astype(str)
    .str.strip()
    .str.upper()
)

print("Provisional terms:", len(provisional_terms))

for term in sorted(provisional_terms):
    print(term)

Provisional terms: 10
ANAPHYLACTIC REACTION
ANAPHYLACTIC SHOCK
ANAPHYLACTOID REACTION
ANAPHYLACTOID SHOCK
DOCUMENTED HYPERSENSITIVITY TO ADMINISTERED DRUG
DRUG HYPERSENSITIVITY
HYPERSENSITIVITY
TYPE I HYPERSENSITIVITY
TYPE II HYPERSENSITIVITY
TYPE IV HYPERSENSITIVITY REACTION


In [14]:
outcome_columns = [
    "primaryid",
    "isr",
    "pt",
    "outcome_concept_id",
    "snomed_outcome_concept_id"
]

In [15]:
def create_report_id(df):
    df = df.copy()

    primary = df["primaryid"].notna()
    legacy = df["isr"].notna()

    df["report_id"] = pd.NA

    df.loc[primary, "report_id"] = (
        "PRIMARY_" + df.loc[primary, "primaryid"]
    )

    df.loc[~primary & legacy, "report_id"] = (
        "ISR_" + df.loc[~primary & legacy, "isr"]
    )

    return df

In [16]:
positive_report_ids = set()
all_report_ids = set()
term_counts = {}

with zipfile.ZipFile(zip_path, "r") as archive:
    with archive.open("standard_case_outcome.tsv") as file:

        for chunk in pd.read_csv(
            file,
            sep="\t",
            header=None,
            names=outcome_columns,
            dtype=str,
            na_values=["\\N"],
            chunksize=100000
        ):
            chunk = create_report_id(chunk)

            chunk["pt_normalized"] = (
                chunk["pt"]
                .astype("string")
                .str.strip()
                .str.upper()
            )

            valid_ids = chunk["report_id"].dropna()
            all_report_ids.update(valid_ids)

            positive_rows = chunk[
                chunk["pt_normalized"].isin(provisional_terms)
            ]

            positive_report_ids.update(
                positive_rows["report_id"].dropna()
            )

            counts = positive_rows["pt_normalized"].value_counts()

            for term, count in counts.items():
                term_counts[term] = term_counts.get(term, 0) + count

print("Total unique reports:", len(all_report_ids))
print("Reports with provisional terms:", len(positive_report_ids))

Total unique reports: 4928344
Reports with provisional terms: 96031


In [17]:
labels_df = pd.DataFrame({
    "report_id": sorted(all_report_ids)
})

labels_df["provisional_hypersensitivity_label"] = (
    labels_df["report_id"]
    .isin(positive_report_ids)
    .astype(int)
)

print("Label distribution:")
print(
    labels_df["provisional_hypersensitivity_label"]
    .value_counts()
)

print("\nLabel percentages:")
print(
    labels_df["provisional_hypersensitivity_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display(labels_df.head())

Label distribution:
provisional_hypersensitivity_label
0    4832313
1      96031
Name: count, dtype: int64

Label percentages:
provisional_hypersensitivity_label
0    98.05
1     1.95
Name: proportion, dtype: float64


,report_id,provisional_hypersensitivity_label
0,ISR_4204616,0
1,ISR_4214534,0
2,ISR_4223542,1
3,ISR_4250482,0
4,ISR_4261823,0


In [18]:
term_counts_df = pd.DataFrame(
    sorted(
        term_counts.items(),
        key=lambda x: x[1],
        reverse=True
    ),
    columns=["adverse_event_term", "outcome_row_count"]
)

display(term_counts_df)

,adverse_event_term,outcome_row_count
0,HYPERSENSITIVITY,43679
1,DRUG HYPERSENSITIVITY,29585
2,ANAPHYLACTIC REACTION,13823
3,ANAPHYLACTIC SHOCK,7700
4,ANAPHYLACTOID REACTION,1896
5,TYPE IV HYPERSENSITIVITY REACTION,576
6,TYPE I HYPERSENSITIVITY,375
7,DOCUMENTED HYPERSENSITIVITY TO ADMINISTERED DRUG,293
8,ANAPHYLACTOID SHOCK,262
9,TYPE II HYPERSENSITIVITY,17


In [19]:
output_dir = Path("/content/drive/MyDrive/aeolus_processed")
output_dir.mkdir(parents=True, exist_ok=True)

labels_df.to_csv(
    output_dir / "provisional_report_labels.csv",
    index=False
)

term_counts_df.to_csv(
    output_dir / "provisional_term_counts.csv",
    index=False
)

print("Files saved successfully.")

Files saved successfully.
